# Module 08 — Logistic Regression & Classification
*Statistics for Econometrics — An intermediate course for researchers*

This module covers logistic regression for binary outcome modeling. We explore why linear regression fails for binary targets, develop the logistic function from first principles, estimate models via maximum likelihood, interpret coefficients as odds ratios, evaluate classification performance, and extend to multiple predictors including categorical variables.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

try:
    import statsmodels.api as sm
    from statsmodels.formula.api import logit
    from statsmodels.genmod.generalized_linear_model import GLM
    from statsmodels.genmod import families
except ImportError:
    !pip install statsmodels
    import statsmodels.api as sm
    from statsmodels.formula.api import logit
    from statsmodels.genmod.generalized_linear_model import GLM
    from statsmodels.genmod import families

try:
    from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score, precision_recall_curve
except ImportError:
    !pip install scikit-learn
    from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score, precision_recall_curve

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
np.random.seed(42)

In [ ]:
# Generate realistic dataset: firm credit default prediction
# 300 firms with financial indicators and default outcome

n = 300
np.random.seed(42)

# Generate continuous predictors
debt_ratio = np.random.uniform(0.1, 2.5, n)  # debt to equity ratio
years_active = np.random.uniform(1, 50, n)  # years in business
log_revenue = np.random.uniform(10, 20, n)  # log of annual revenue

# Generate categorical predictor
industry = np.random.choice(['manufacturing', 'services', 'tech'], n)
is_services = (industry == 'services').astype(int)
is_tech = (industry == 'tech').astype(int)

# Generate default probability via logistic function
# Probability depends on: debt_ratio (increases risk), years_active (decreases risk),
# log_revenue (decreases risk), industry effects

linear_pred = -3.5 + 1.2*debt_ratio - 0.05*years_active - 0.15*log_revenue + 0.8*is_services + 1.5*is_tech
default_prob = 1 / (1 + np.exp(-linear_pred))  # Logistic function

# Generate binary outcome
default = (np.random.random(n) < default_prob).astype(int)

# Create dataframe
data = pd.DataFrame({
    'debt_ratio': debt_ratio,
    'years_active': years_active,
    'log_revenue': log_revenue,
    'industry': industry,
    'default': default
})

print(data.head(10))
print(f"\nDataset shape: {data.shape}")
print(f"\nDefault rate: {default.mean():.3f}")
print(f"\nSummary statistics:")
print(data.describe())

## 8.1 — Why Linear Regression Fails

When the outcome is binary (0/1), ordinary least squares regression has critical limitations:
- **Predicted values outside [0,1]**: Linear model produces predictions like 1.2 or -0.3, which are nonsensical for probabilities
- **Heteroscedasticity**: Error variance depends on fitted probability, violating Gauss-Markov assumptions
- **Non-normality of errors**: Binary outcomes cannot be normally distributed

Logistic regression addresses these by modeling the probability directly via a bounded function.

In [ ]:
# Fit OLS regression on binary outcome (wrong approach)
from statsmodels.formula.api import ols

ols_model = ols('default ~ debt_ratio + years_active + log_revenue', data=data).fit()

print("="*80)
print("OLS REGRESSION ON BINARY OUTCOME (Incorrect)")
print("="*80)
print(ols_model.summary().tables[1])

# Examine predicted values
ols_pred = ols_model.predict()

print(f"\nOLS Predicted values:")
print(f"  Min: {ols_pred.min():.4f}")
print(f"  Max: {ols_pred.max():.4f}")
print(f"  Values < 0: {(ols_pred < 0).sum()}")
print(f"  Values > 1: {(ols_pred > 1).sum()}")

# Plot OLS fit
fig, ax = plt.subplots(figsize=(12, 6))

# Scatter plot of actual outcomes
ax.scatter(data['debt_ratio'], data['default'], alpha=0.5, s=60, label='Actual (0/1)')

# OLS fitted line
sorted_idx = np.argsort(data['debt_ratio'])
ax.plot(data['debt_ratio'].iloc[sorted_idx], ols_pred.iloc[sorted_idx], 
        'r-', linewidth=2, label='OLS fitted line')

ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.axhline(y=1, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Debt Ratio', fontsize=12)
ax.set_ylabel('Default Probability', fontsize=12)
ax.set_title('Linear Regression on Binary Outcome: Predictions Outside [0,1]', 
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8.2 — The Logistic Function

The **logistic function** maps any real number to the interval (0,1):

$$P(Y=1|X) = \frac{e^{\beta_0 + \beta_1 X}}{1 + e^{\beta_0 + \beta_1 X}} = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X)}}$$

This S-shaped curve has desirable properties:
- Always produces probabilities in (0,1)
- Symmetric around 0.5
- Steeper slope at p=0.5, flatter at extremes
- Behavior controlled by coefficients: larger β₁ = steeper slope

In [ ]:
# Plot sigmoid function for different coefficient values
x = np.linspace(-6, 6, 100)

# Different scenarios
beta_scenarios = [
    {'beta_0': 0, 'beta_1': 0.5, 'label': 'β₁ = 0.5 (shallow)'},
    {'beta_0': 0, 'beta_1': 1.0, 'label': 'β₁ = 1.0 (moderate)'},
    {'beta_0': 0, 'beta_1': 2.0, 'label': 'β₁ = 2.0 (steep)'},
    {'beta_0': -1, 'beta_1': 1.0, 'label': 'β₀ = -1, β₁ = 1.0 (shifted left)'}
]

fig, ax = plt.subplots(figsize=(12, 6))

for scenario in beta_scenarios:
    y = 1 / (1 + np.exp(-(scenario['beta_0'] + scenario['beta_1']*x)))
    ax.plot(x, y, linewidth=2.5, label=scenario['label'])

ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.axvline(x=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('X (linear predictor scale)', fontsize=12)
ax.set_ylabel('P(Y=1|X)', fontsize=12)
ax.set_title('Logistic Function (Sigmoid) for Different Coefficients', 
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Show connection between probability, odds, and log-odds
prob = np.array([0.1, 0.25, 0.5, 0.75, 0.9])
odds = prob / (1 - prob)
log_odds = np.log(odds)

results_table = pd.DataFrame({
    'Probability': prob,
    'Odds': odds.round(4),
    'Log-Odds': log_odds.round(4)
})

print("="*70)
print("RELATIONSHIP: Probability ↔ Odds ↔ Log-Odds")
print("="*70)
print(results_table.to_string(index=False))
print(f"\nKey insight:")
print(f"  - Logistic regression models log-odds as linear function of X")
print(f"  - Coefficients β directly interpret as change in log-odds")
print(f"  - exp(β) gives odds ratio (proportional change in odds)")

## 8.3 — Fitting the Model: Maximum Likelihood

Unlike OLS which minimizes sum of squared residuals, logistic regression estimates coefficients by **maximum likelihood (ML)**. The likelihood function for binary outcomes is:

$$L(\beta) = \prod_{i=1}^{n} P_i^{y_i} (1-P_i)^{1-y_i}$$

where $P_i = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X_i)}}$

ML finds coefficients that maximize this likelihood. Statsmodels uses numerical optimization to solve this non-linear problem.

In [ ]:
# Fit logistic regression using statsmodels
logit_model = sm.Logit.from_formula('default ~ debt_ratio + years_active + log_revenue', 
                                      data=data).fit(disp=0)

print("="*80)
print("LOGISTIC REGRESSION SUMMARY")
print("="*80)
print(logit_model.summary())

print(f"\nLog-Likelihood: {logit_model.llf:.4f}")
print(f"AIC: {logit_model.aic:.4f}")
print(f"BIC: {logit_model.bic:.4f}")

## 8.4 — Interpreting Coefficients: Odds Ratios

Logistic regression coefficients are on the **log-odds scale**. To interpret them intuitively:

- **β directly**: A one-unit increase in X increases log-odds by β (difficult to interpret)
- **exp(β) = Odds Ratio**: A one-unit increase in X multiplies the odds by exp(β)
  - exp(β) = 1.2: 20% increase in odds
  - exp(β) = 0.8: 20% decrease in odds

Note: Odds ratios are NOT the same as probability ratios, but they approximate them when probabilities are small.

In [ ]:
# Extract and display odds ratios
coefficients = logit_model.params
odds_ratios = np.exp(coefficients)
se = logit_model.bse
conf_int = logit_model.conf_int()

# Odds ratios for confidence intervals
or_ci_lower = np.exp(conf_int.iloc[:, 0])
or_ci_upper = np.exp(conf_int.iloc[:, 1])

or_table = pd.DataFrame({
    'Coefficient': coefficients.round(6),
    'Odds Ratio': odds_ratios.round(6),
    'OR_95%_CI_Lower': or_ci_lower.round(6),
    'OR_95%_CI_Upper': or_ci_upper.round(6)
})

print("="*80)
print("COEFFICIENTS AND ODDS RATIOS")
print("="*80)
print(or_table)

print(f"\nInterpretation:")
for var in coefficients.index:
    or_val = odds_ratios[var]
    pct_change = (or_val - 1) * 100
    print(f"  {var}: OR = {or_val:.4f}")
    if pct_change > 0:
        print(f"    → One-unit increase → {pct_change:.2f}% higher odds of default")
    else:
        print(f"    → One-unit increase → {abs(pct_change):.2f}% lower odds of default")

In [ ]:
# Marginal effects at the mean
margeff = logit_model.get_margeff()

print("="*80)
print("MARGINAL EFFECTS AT THE MEAN")
print("="*80)
print(margeff.summary())

print(f"\nInterpretation:")
print(f"Marginal effect = change in probability for one-unit increase in X")
print(f"Computed at mean values of all predictors")

## 8.5 — Predicted Probabilities

Once we have estimated coefficients, we can compute predicted probabilities for any observation. The model produces a probability between 0 and 1 for each firm based on its characteristics.

In [ ]:
# Compute predicted probabilities for all observations
pred_prob = logit_model.predict(data[['debt_ratio', 'years_active', 'log_revenue']])

print(f"Predicted probabilities:")
print(f"  Min: {pred_prob.min():.4f}")
print(f"  Max: {pred_prob.max():.4f}")
print(f"  Mean: {pred_prob.mean():.4f}")
print(f"  All in [0,1]: {((pred_prob >= 0) & (pred_prob <= 1)).all()}")

# Plot distribution of predicted probabilities
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of predicted probabilities
ax = axes[0]
ax.hist(pred_prob, bins=30, alpha=0.7, edgecolor='black')
ax.axvline(x=pred_prob.mean(), color='r', linestyle='--', linewidth=2, 
           label=f'Mean = {pred_prob.mean():.3f}')
ax.set_xlabel('Predicted Probability', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Distribution of Predicted Default Probabilities', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Predicted prob vs actual outcome
ax = axes[1]
ax.scatter(pred_prob[data['default']==0], np.zeros(sum(data['default']==0)) + np.random.normal(0, 0.02, sum(data['default']==0)), 
          alpha=0.5, s=50, label='Did not default (Y=0)')
ax.scatter(pred_prob[data['default']==1], np.ones(sum(data['default']==1)) + np.random.normal(0, 0.02, sum(data['default']==1)), 
          alpha=0.5, s=50, label='Defaulted (Y=1)')
ax.axvline(x=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Predicted Probability', fontsize=11)
ax.set_ylabel('Actual Outcome', fontsize=11)
ax.set_title('Predicted Probability vs Actual Outcome', fontsize=12, fontweight='bold')
ax.set_yticks([0, 1])
ax.set_yticklabels(['No', 'Yes'])
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compute predicted probabilities at specific values
test_scenarios = pd.DataFrame({
    'debt_ratio': [0.5, 1.0, 1.5, 2.0],
    'years_active': [10, 10, 10, 10],
    'log_revenue': [15, 15, 15, 15]
})

test_pred_prob = logit_model.predict(test_scenarios)
test_scenarios['pred_default_prob'] = test_pred_prob

print("="*70)
print("PREDICTED PROBABILITIES AT SPECIFIC VALUES")
print("="*70)
print(f"\nFirms with 10 years active, log(revenue)=15, varying debt ratios:")
print(test_scenarios.to_string(index=False))

## 8.6 — Goodness of Fit

Unlike R² in OLS, there is no single perfect goodness-of-fit measure for logistic regression. Common alternatives:

- **McFadden's Pseudo-R²**: $1 - \frac{LL_\text{full}}{LL_\text{null}}$ (compares to null model with only intercept)
- **AIC / BIC**: Information criteria for model comparison (lower is better)
- **Log-Likelihood Ratio Test**: Compares nested models

Pseudo-R² typically ranges from 0 to 1 but rarely reaches 0.6 even for good models.

In [ ]:
# Compute goodness-of-fit measures
print("="*80)
print("GOODNESS OF FIT MEASURES")
print("="*80)

# McFadden's Pseudo-R²
ll_null = sm.Logit.from_formula('default ~ 1', data=data).fit(disp=0).llf
ll_full = logit_model.llf
mcfadden_r2 = 1 - (ll_full / ll_null)

print(f"\nLog-Likelihood (Full Model): {ll_full:.4f}")
print(f"Log-Likelihood (Null Model): {ll_null:.4f}")
print(f"\nMcFadden's Pseudo-R²: {mcfadden_r2:.4f}")
print(f"AIC: {logit_model.aic:.4f}")
print(f"BIC: {logit_model.bic:.4f}")

# Log-likelihood ratio test
lr_stat = 2 * (ll_full - ll_null)
lr_pval = 1 - stats.chi2.cdf(lr_stat, df=3)  # 3 predictors

print(f"\nLog-Likelihood Ratio Test (Model vs Null):")
print(f"  LR Statistic: {lr_stat:.4f}")
print(f"  Degrees of freedom: 3")
print(f"  P-value: {lr_pval:.6f}")
if lr_pval < 0.05:
    print(f"  → Model is significantly better than null model (p < 0.05)")
else:
    print(f"  → Model is NOT significantly better than null model (p >= 0.05)")

## 8.7 — Classification and Confusion Matrix

To use logistic regression for classification, we must choose a **decision threshold** (typically 0.5). Predicted probabilities above the threshold are classified as 1, below as 0.

The **confusion matrix** summarizes classification performance:
- **True Positives (TP)**: Correctly predicted defaults
- **False Positives (FP)**: Incorrectly predicted defaults
- **True Negatives (TN)**: Correctly predicted non-defaults
- **False Negatives (FN)**: Missed defaults (Type II error)

Key metrics derived from the confusion matrix:
- **Accuracy**: (TP + TN) / (TP + TN + FP + FN) — overall correctness
- **Precision**: TP / (TP + FP) — of predicted positives, how many are correct?
- **Recall (Sensitivity)**: TP / (TP + FN) — of actual positives, how many are caught?
- **F1 Score**: Harmonic mean of precision and recall

In [ ]:
# Confusion matrix at threshold 0.5
threshold = 0.5
pred_class = (pred_prob >= threshold).astype(int)
actual = data['default'].values

cm = confusion_matrix(actual, pred_class)

print("="*70)
print(f"CONFUSION MATRIX (Threshold = {threshold})")
print("="*70)
print(f"\n                Predicted")
print(f"                No    Yes")
print(f"Actual No     {cm[0,0]:4d}  {cm[0,1]:4d}")
print(f"       Yes    {cm[1,0]:4d}  {cm[1,1]:4d}")

tn, fp, fn, tp = cm.ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\nClassification metrics:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")

In [ ]:
# Classification metrics at multiple thresholds
thresholds = [0.3, 0.5, 0.7]

metrics_table = []
for thresh in thresholds:
    pred_class = (pred_prob >= thresh).astype(int)
    cm = confusion_matrix(actual, pred_class)
    tn, fp, fn, tp = cm.ravel()
    
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    metrics_table.append({
        'Threshold': thresh,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'Specificity': specificity,
        'F1': f1
    })

metrics_df = pd.DataFrame(metrics_table)

print("="*90)
print("CLASSIFICATION METRICS AT DIFFERENT THRESHOLDS")
print("="*90)
print(metrics_df.to_string(index=False))

print(f"\nTradeoff: Higher threshold → Higher precision, lower recall")
print(f"(More conservative predictions, fewer false positives, more false negatives)")

## 8.8 — ROC Curve and AUC

The **ROC (Receiver Operating Characteristic) curve** plots the tradeoff between:
- **True Positive Rate (Sensitivity/Recall)**: TP / (TP + FN)
- **False Positive Rate (1 - Specificity)**: FP / (FP + TN)

Each point on the ROC curve corresponds to a different classification threshold. The curve sweeps from (0,0) to (1,1) as the threshold decreases from 1 to 0.

The **AUC (Area Under the Curve)** summarizes ROC performance in a single number:
- AUC = 1: Perfect classifier
- AUC = 0.5: No better than random guessing
- AUC = 0: Completely wrong predictions (reverse of perfect)

AUC is threshold-independent and preferred to accuracy when classes are imbalanced.

In [ ]:
# Compute ROC curve
fpr, tpr, roc_thresholds = roc_curve(actual, pred_prob)
roc_auc = auc(fpr, tpr)

print(f"AUC (Area Under ROC Curve): {roc_auc:.4f}")

# Plot ROC curve
fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(fpr, tpr, linewidth=2.5, label=f'ROC Curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier (AUC = 0.5)')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Precision-recall tradeoff at different thresholds
precision_vals, recall_vals, pr_thresholds = precision_recall_curve(actual, pred_prob)

fig, ax = plt.subplots(figsize=(11, 7))

ax.plot(recall_vals, precision_vals, linewidth=2.5, label='Precision-Recall Curve')

# Mark specific thresholds
for thresh in [0.3, 0.5, 0.7]:
    idx = np.argmin(np.abs(pr_thresholds - thresh))
    ax.scatter(recall_vals[idx], precision_vals[idx], s=100, zorder=5, 
              label=f'Threshold = {thresh}')

ax.set_xlabel('Recall (True Positive Rate)', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Tradeoff', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

print(f"\nPrecision-recall tradeoff:")
print(f"  - Higher threshold → Higher precision (fewer false positives)")
print(f"  - Lower threshold → Higher recall (catch more true positives)")

## 8.9 — Multiple Logistic Regression with Categorical Predictors

Logistic regression extends naturally to multiple predictors, both continuous and categorical. For categorical variables, we use **dummy encoding** (one-hot encoding with reference category omitted).

The reference category is implicitly at log-odds = 0, so we interpret other categories relative to the reference.

In [ ]:
# Fit model with industry dummies (manufacturing as reference)
data_with_dummies = data.copy()
data_with_dummies['is_services'] = (data['industry'] == 'services').astype(int)
data_with_dummies['is_tech'] = (data['industry'] == 'tech').astype(int)

logit_full = sm.Logit.from_formula(
    'default ~ debt_ratio + years_active + log_revenue + is_services + is_tech',
    data=data_with_dummies
).fit(disp=0)

print("="*80)
print("LOGISTIC REGRESSION WITH CATEGORICAL PREDICTORS")
print("="*80)
print(logit_full.summary())

In [ ]:
# Odds ratios by industry
coef_full = logit_full.params
or_full = np.exp(coef_full)
se_full = logit_full.bse
conf_int_full = logit_full.conf_int()
or_ci_lower_full = np.exp(conf_int_full.iloc[:, 0])
or_ci_upper_full = np.exp(conf_int_full.iloc[:, 1])

or_table_full = pd.DataFrame({
    'Variable': coef_full.index,
    'Coefficient': coef_full.values.round(6),
    'Odds_Ratio': or_full.values.round(6),
    'OR_95%_CI_Lower': or_ci_lower_full.values.round(6),
    'OR_95%_CI_Upper': or_ci_upper_full.values.round(6)
})

print("="*100)
print("COEFFICIENTS AND ODDS RATIOS (WITH INDUSTRY DUMMIES)")
print("="*100)
print(or_table_full.to_string(index=False))

print(f"\nInterpretation of industry effects (reference = Manufacturing):")
or_services = or_full['is_services']
or_tech = or_full['is_tech']
print(f"  Services firms: {(or_services-1)*100:.2f}% {'higher' if or_services > 1 else 'lower'} odds of default")
print(f"  Tech firms:     {(or_tech-1)*100:.2f}% {'higher' if or_tech > 1 else 'lower'} odds of default")
print(f"  (Compared to manufacturing firms at same levels of other predictors)")